In [5]:
import pandas as pd
import json
from datetime import datetime, timedelta
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import pytz
import requests
import psycopg2

In [ ]:
def read_db_credentials(path="data/config.txt"):
    creds = {}
    with open(path, "r") as f:
        for line in f:
            key, value = line.strip().split("=")
            creds[key] = value
    return creds

def connect_to_db(creds):
    return psycopg2.connect(
        host=creds["host"],
        port=creds["port"],
        dbname=creds["database"],
        user=creds["user"],
        password=creds["password"]
    )

In [14]:
def read_db_credentials(path="data/config.txt"):
    creds = {}
    with open(path, "r") as f:
        for line in f:
            key, value = line.strip().split("=")
            creds[key] = value
    return creds

def connect_to_db(creds):
    return psycopg2.connect(
        host=creds["host"],
        port=creds["port"],
        dbname=creds["database"],
        user=creds["user"],
        password=creds["password"]
    )
def format_columns(cols):
    if isinstance(cols, (list, tuple)):
        return ",".join(map(str, cols))
    return str(cols)

def request_user_data(user_number, cols, data):
    creds = read_db_credentials()
    conn = connect_to_db(creds)

    # Formatierung der Spaltenliste (egal ob Liste oder Einzelwert)
    if isinstance(cols, (list, tuple)):
        cols_str = ",".join(map(str, cols))
    else:
        cols_str = str(cols)

    query = f"""
        SELECT {cols_str}
        FROM fact_raw_data
        WHERE user_number = %s AND data_source = %s;
    """

    # Sicher ausführen mit Parametern
    cursor = conn.cursor()
    cursor.execute(query, (user_number, data))
    rows = cursor.fetchall()
    cursor.close()
    conn.close()
    
    return rows
    
    


In [15]:
spotify_raw = request_user_data(1,"raw_json","spotify")

In [ ]:
import json

def strip_structure(obj):
    if isinstance(obj, dict):
        return {k: strip_structure(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        if obj:
            return [strip_structure(obj[0])]
        else:
            return []
    else:
        return None

# ⚠️ Dein Beispiel: Liste mit einem Tupel, das ein Dict enthält
data = [({'spotify_json': {
    'followerCount': 0,
    'followingUsersCount': 0,
    'dismissingUsersCount': 0,
    'identifierType': 'email',
    'identifierValue': 'test@example.com',
    'playlists': [
        {'name': 'Pop', 'tracks': 42}
    ]
}})]

# Zugriff auf das Dict im Tupel
cleaned = strip_structure(spotify_raw[0][0])

# Ausgabe schön formatiert
print(json.dumps(cleaned, indent=2))


{
  "spotify_json": {
    "followerCount": null,
    "followingUsersCount": null,
    "dismissingUsersCount": null,
    "identifierType": null,
    "identifierValue": null,
    "displayName": null,
    "firstName": null,
    "lastName": null,
    "imageUrl": null,
    "largeImageUrl": null,
    "tasteMaker": null,
    "verified": null,
    "username": null,
    "email": null,
    "country": null,
    "createdFromFacebook": null,
    "facebookUid": null,
    "birthdate": null,
    "gender": null,
    "postalCode": null,
    "mobileNumber": null,
    "mobileOperator": null,
    "mobileBrand": null,
    "creationTime": null,
    "assuredEstimatedAge": null,
    "assuredAgeMethod": null,
    "assuredAgeTimestamp": null,
    "tracks": [],
    "albums": [],
    "shows": [],
    "episodes": [],
    "bannedTracks": [],
    "artists": [
      {
        "name": null,
        "uri": null
      }
    ],
    "bannedArtists": [],
    "other": []
  },
  "Follow": {
    "followerCount": null,
    "fol

In [45]:
import requests
import base64

client_id = '8541e8dfc0fd4a86916d0d98cdb150ad'
client_secret = 'cade2256b5804831a11349f86e6a1784'

import requests
import base64
import time

class SpotifyAuth:
    def __init__(self, client_id, client_secret):
        self.client_id = client_id
        self.client_secret = client_secret
        self.access_token = None
        self.token_expires_at = 0  # Unix timestamp

    def _fetch_token(self):
        auth_str = f"{self.client_id}:{self.client_secret}"
        b64_auth_str = base64.b64encode(auth_str.encode()).decode()

        response = requests.post(
            "https://accounts.spotify.com/api/token",
            data={"grant_type": "client_credentials"},
            headers={"Authorization": f"Basic {b64_auth_str}"}
        )

        if response.status_code == 200:
            token_data = response.json()
            self.access_token = token_data["access_token"]
            # Spotify gibt Gültigkeit in Sekunden an (meist 3600)
            self.token_expires_at = time.time() + token_data.get("expires_in", 3600) - 60
        else:
            raise Exception(f"Fehler beim Abrufen des Tokens: {response.status_code} - {response.text}")


    def get_token(self):
        if not self.access_token or time.time() >= self.token_expires_at:
            self._fetch_token()
        return self.access_token



In [ ]:
from collections import Counter
import requests
auth = SpotifyAuth(client_id, client_secret)

from collections import Counter
import requests

def get_weighted_genres(track_query, target_artist):

    access_token = auth.get_token()
    headers = {"Authorization": f"Bearer {access_token}"}
    params = {"q": track_query, "type": "track", "limit": 20}
    response = requests.get("https://api.spotify.com/v1/search", headers=headers, params=params)
    data = response.json()

    genre_counter = Counter()
    total_artist_matches = 0

    for track in data.get("tracks", {}).get("items", []):
        matched_artist = next(
            (artist for artist in track["artists"] if artist["name"].lower() == target_artist.lower()), None
        )
        if matched_artist:
            total_artist_matches += 1
            artist_id = matched_artist["id"]
            artist_data = requests.get(f"https://api.spotify.com/v1/artists/{artist_id}", headers=headers).json()
            genres = artist_data.get("genres", [])
            genre_counter.update(genres)

    if total_artist_matches == 0 or not genre_counter:
        return {}

    # Gesamtanzahl an Genre-Zuordnungen
    total_genre_mentions = sum(genre_counter.values())

    # Normalisierte Gewichtung (Summe == 1)
    normalized_genres = {
        genre: round(count / total_genre_mentions, 3)
        for genre, count in genre_counter.items()
    }

    return normalized_genres



In [48]:
get_weighted_genres(title = "silver springs", artist_name="Fleetwood Mac")

{'classic rock': 1.0, 'yacht rock': 1.0, 'soft rock': 1.0}